# Shared BQ2 functions

Common loading, psychrometric and AC-event definitions for Tracks A/B. Use the setup cell in an analysis notebook with `RUN_CHECKS = False`; no private tests are required to load these functions. Coordinate definition changes across tracks.

Inputs: cleaned hourly/minute tables and the existing ERA5 cache. See the [README](../README.md) for setup and notebook order. The legacy F5-Z1 flag text describes a supply-air **hypothesis**; placement and calibration have not been verified.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RUN_CHECKS = globals().get("RUN_CHECKS", True)   # set RUN_CHECKS = False before %run -i to skip Section 6

## Scope and flags

Floors 3–7, zones 1, 2, 4 and 5 (20 zones). Other floors/zones lack comparable sensors or layout. Retain F5-Z1 with an environmental QA flag, not an ordinary ranking. Environmental outages vary by channel; check actual coverage rather than assuming a whole season is usable.

In [2]:
SCOPE_FLOORS = [3, 4, 5, 6, 7]
SCOPE_ZONES = [1, 2, 4, 5]

SUSPECT_SENSORS = {(5, 1): "Sensor in AC supply air: temperature biased cold, RH biased high (cleaning notebook 3.7/6.5)"}

SENSOR_OUTAGE = (pd.Timestamp("2018-09-12"), pd.Timestamp("2019-03-06"))

## Psychrometrics

RH depends on temperature; humidity ratio expresses water vapour per kilogram of dry air. The functions provide dew point (°C), humidity ratio (g/kg), moist-air enthalpy (kJ/kg) and assumed-surface RH (%), using Magnus coefficients and 1013.25 hPa.

Assumed surface conditions are scenarios, not measurements of condensation or mould. Enthalpy is weather context, not measured AC cooling output.

In [3]:
def sat_vp_hpa(t_c):
    """Saturation vapour pressure (hPa), Magnus formula (Sonntag 1990 coefficients)."""
    return 6.112 * np.exp(17.62 * t_c / (243.12 + t_c))


def dew_point_c(t_c, rh_pct):
    """Dew point (°C) from air temperature (°C) and relative humidity (%)."""
    rh = np.clip(rh_pct, 1, 100)
    g = np.log(rh / 100) + 17.62 * t_c / (243.12 + t_c)
    return 243.12 * g / (17.62 - g)


def humidity_ratio_gkg(t_c, rh_pct, p_hpa=1013.25):
    """Grams of water vapour per kg of dry air."""
    pv = np.clip(rh_pct, 0, 100) / 100 * sat_vp_hpa(t_c)
    return 622 * pv / (p_hpa - pv)


def enthalpy_kjkg(t_c, w_gkg):
    """Moist-air enthalpy (kJ per kg dry air). Captures sensible + latent load in one number."""
    w = w_gkg / 1000
    return 1.006 * t_c + w * (2501 + 1.86 * t_c)


def surface_rh_pct(dew_c, surface_t_c):
    """RH at a surface colder than the room air (same moisture content, lower temperature)."""
    return np.clip(100 * sat_vp_hpa(dew_c) / sat_vp_hpa(surface_t_c), 0, 100)

## Seasonal labels

Project month groups: hot March–May, rainy June–October, cool November–February. They are approximate categories; comparisons still require actual weather and adequate coverage.

In [4]:
SEASON_ORDER = ["hot", "rainy", "cool"]


def season(month):
    """Hot: Mar-May, rainy: Jun-Oct, cool: Nov-Feb. Works on an int or a Series of months."""
    m = pd.Series(month) if not isinstance(month, pd.Series) else month
    out = pd.Series(np.select([m.between(3, 5), m.between(6, 10)], ["hot", "rainy"], "cool"), index=m.index)
    return pd.Categorical(out, categories=SEASON_ORDER, ordered=True)

## Loading and weather merge

`load_zone_hourly` filters scope and adds indoor features/flags. `merge_weather` adds outdoor features and the legacy `moisture_removal_gkg` field: outdoor minus indoor humidity ratio. That gap is a supporting indicator, not measured coil removal. Load the existing cache; do not refresh upstream data just to run BQ2.

In [5]:
def load_zone_hourly(path, scope_only=True):
    """Cleaned zone x hour table + indoor dew point, humidity ratio, season and quality flag."""
    zh = pd.read_csv(path, parse_dates=["Date"])
    if scope_only:
        zh = zh[zh["floor"].isin(SCOPE_FLOORS) & zh["zone"].astype(int).isin(SCOPE_ZONES)].copy()
    zh["zone"] = zh["zone"].astype(int)
    zh["indoor_dew_C"] = dew_point_c(zh["temp_C"], zh["rh_pct"])
    zh["indoor_w_gkg"] = humidity_ratio_gkg(zh["temp_C"], zh["rh_pct"])
    zh["season"] = season(zh["Date"].dt.month)
    zh["year"] = zh["Date"].dt.year
    zh["suspect_sensor"] = [(f, z) in SUSPECT_SENSORS for f, z in zip(zh["floor"], zh["zone"])]
    return zh


def add_outdoor_psychrometrics(weather):
    """Adds outdoor humidity ratio and enthalpy to the ERA5 table from load_bangkok_era5_weather()."""
    w = weather.copy()
    w["outdoor_w_gkg"] = humidity_ratio_gkg(w["outdoor_temp_C"], w["outdoor_rh_pct"])
    w["outdoor_enthalpy_kJkg"] = enthalpy_kjkg(w["outdoor_temp_C"], w["outdoor_w_gkg"])
    return w


def merge_weather(zone_hourly, weather):
    """Joins hourly ERA5 onto the zone table by timestamp and adds moisture removal (out - in)."""
    w = add_outdoor_psychrometrics(weather)
    w = w.reset_index().rename(columns={w.index.name or "index": "Date"})
    df = zone_hourly.merge(w, on="Date", how="left")
    df["moisture_removal_gkg"] = df["outdoor_w_gkg"] - df["indoor_w_gkg"]
    return df

## Power-defined AC events

On = power above max(0.5 kW, 20% of the zone's 95th-percentile power). The detector finds candidate transitions; validate timestamp continuity and joint sensor coverage separately before calculating startup or rebound metrics. Missing readings must not create observed continuous exposure.

Startup/shutdown responses also depend on initial conditions, controls, loads and surrounding spaces. They are observational comparisons, not isolated capacity tests or randomised experiments. `zone_minute_series` requires all assigned AC meters when summing zone power.

In [6]:
def ac_on_threshold(ac_kw):
    """'On' = above 20% of the zone's 95th-percentile AC power, and at least 0.5 kW (above standby)."""
    return max(0.5, 0.2 * ac_kw.quantile(0.95))


def detect_ac_events(ac_kw, min_state_minutes=30):
    """Start-ups and shutdowns from a minute-level AC kW series (DatetimeIndex).

    An event is kept only if the AC was in the old state for >= min_state_minutes before it and
    stays in the new state for >= min_state_minutes after it, so brief cycling isn't counted.
    Returns a DataFrame: time, event ('start' or 'stop'), threshold_kW.
    """
    ac_kw = ac_kw.dropna()
    thr = ac_on_threshold(ac_kw)
    on = (ac_kw > thr).astype(int)
    run_id = on.ne(on.shift()).cumsum()
    runs = pd.DataFrame({"state": on.groupby(run_id).first(),
                         "start": ac_kw.index.to_series().groupby(run_id).first(),
                         "minutes": on.groupby(run_id).size()}).reset_index(drop=True)
    events = []
    for i in range(1, len(runs)):
        prev, cur = runs.iloc[i - 1], runs.iloc[i]
        if prev.minutes < min_state_minutes or cur.minutes < min_state_minutes:
            continue
        before = cur.start - pd.Timedelta(minutes=1)
        if before not in ac_kw.index:      # a data gap, not a real switch
            continue
        events.append({"time": cur.start, "event": "start" if cur.state == 1 else "stop"})
    out = pd.DataFrame(events, columns=["time", "event"])
    out["threshold_kW"] = thr
    return out


def zone_minute_series(clean_dir, floor, zone):
    """Minute-level AC kW (all AC meters summed), temperature and RH for one zone, both years."""
    parts = [pd.read_csv(Path(clean_dir) / f"{y}Floor{floor}.csv", parse_dates=["Date"], index_col="Date")
             for y in (2018, 2019)]
    df = pd.concat(parts)
    ac_cols = [c for c in df.columns if c.startswith(f"z{zone}_AC")]
    out = pd.DataFrame(index=df.index)
    out["ac_kW"] = df[ac_cols].sum(axis=1, min_count=len(ac_cols)) if ac_cols else np.nan
    out["temp_C"] = df.get(f"z{zone}_S1(degC)")
    out["rh_pct"] = df.get(f"z{zone}_S1(RH%)")
    out["indoor_w_gkg"] = humidity_ratio_gkg(out["temp_C"], out["rh_pct"])
    return out